# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants
load_dotenv(override=True)
gemini_api_key = os.getenv('GEMINI_API_KEY')

# Gemini (this course also uses GOOGLE_API_KEY)
if not gemini_api_key:
    print("GEMINI_API_KEY is missing — check the name in .env")
elif not gemini_api_key.startswith(("AIz", "AQ.")):
    print("GEMINI_API_KEY is set, but does not start with AIz or AQ.")
else:
    print(f"GEMINI_API_KEY loaded, starts with {gemini_api_key[:4]}")

MODEL = 'gemini-3.1-flash-lite'
gemini = OpenAI(
    base_url='https://generativelanguage.googleapis.com/v1beta/openai/', 
    api_key=gemini_api_key
)

GEMINI_API_KEY loaded, starts with AIza


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.1-flash-lite
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'profile page', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
NEW
Microduck: A Tiny Robot for AI Builders 🦆
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-Flash-Next
Updated
5 days ago
•
159k
•
4.52k
zai-org/GLM-5.3-Flash
Updated
about 12 hours ago
•
379k
•
1.81k
zai-org/GLM-5.3
Updated
about 10 hours a

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nNEW\nMicroduck: A Tiny Robot for AI Builders 🦆\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\

In [19]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Hugging Face: The AI Community Building the Future

### About the Company
Hugging Face is the leading collaboration platform for the global machine learning community. Operating as the central hub for AI innovation, Hugging Face provides the infrastructure—including models, datasets, and applications—that allows researchers, developers, and organizations to work together to shape the future of artificial intelligence.

### Our Mission
We believe in democratizing AI. By providing an open-source-first environment, we enable builders to discover, share, and deploy cutting-edge machine learning technology. From individual hobbyists to massive enterprises, we provide the tools to turn complex data into actionable intelligence.

### What We Offer
*   **Massive Repository:** Access over 2 million models, 500,000 datasets, and 1 million AI applications (Spaces).
*   **Collaboration Tools:** A seamless platform to create, host, and iterate on machine learning projects with a global community.
*   **Enterprise Solutions:** Robust infrastructure, including Inference Endpoints, Enterprise Support, and AI-native object storage, tailored to scale AI within any organization.
*   **Continuous Learning:** A wealth of resources including blog posts, technical articles, and learning tracks (such as Hugging Face Fundamentals) to keep the community at the forefront of AI development.

### Our Culture
Hugging Face is built on the spirit of openness, collaboration, and rapid innovation. We act as a home for the AI community—a place where the latest breakthroughs are debated, shared, and implemented. Whether it’s tackling the ethics of AI, advancing agentic research, or managing global compute landscapes, our culture is defined by transparency and a shared passion for advancing technology for the benefit of all.

### Join the Community
*   **For Users:** Explore millions of pre-trained models and datasets to jumpstart your next project.
*   **For Investors:** Partner with the industry standard for open-source AI, powering the next generation of technological infrastructure.
*   **For Talent:** We are constantly seeking passionate individuals to help us build the foundational layers of the AI ecosystem. Join us on our journey to make machine learning accessible and collaborative for everyone.

**Discover more at:** [huggingface.co](https://huggingface.co)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True) # initialize empty markdown display and get a handle to update it
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the central collaboration platform for the global machine learning community. Positioned at the heart of the AI revolution, we empower engineers, scientists, and end users to share, discover, and experiment with open-source machine learning to build an ethical and open AI future.

## Our Platform
We provide the infrastructure for the next generation of AI development. Our Hub serves as a massive, open-source repository where users can access:
* **2M+ Models:** Pre-trained and fine-tuned models for virtually any task.
* **500k+ Datasets:** Comprehensive resources to train and refine AI models.
* **1M+ Applications (Spaces):** Interactive AI apps and demos built by our community.
* **Advanced Infrastructure:** Including Inference Endpoints, Storage Buckets, and enterprise-grade support.

## For Customers & Enterprises
Hugging Face offers specialized solutions for organizations looking to integrate AI into their workflows. From Enterprise Support to dedicated infrastructure, our platform allows teams to build securely and scale efficiently. Whether you are experimenting with new models or deploying production-ready AI applications, our suite of enterprise tools ensures you have the support and hardware necessary to succeed.

## Our Culture
At Hugging Face, we believe in the power of open source. Our culture is defined by:
* **Collaboration:** We provide a central space where the global community can work together on complex problems.
* **Innovation:** Our world-class science team is constantly exploring the edge of technology, ensuring we remain at the forefront of AI research.
* **Open & Ethical Standards:** We are committed to fostering an AI future that is transparent, accessible, and ethically developed.

## Careers
We are always looking for passionate individuals to join our mission of building the future of machine learning. By joining Hugging Face, you become part of a fast-growing, highly technical team that values creativity, scientific inquiry, and community impact. 

Visit our website to explore our current openings and help us build the tools that empower the global AI community.

***

**Get Started**
Join the movement today at [huggingface.co](https://huggingface.co). Explore our models, participate in the community, and start building the future with us.

In [23]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


# Hugging Face: Building the Future of AI, Together

## Who We Are
Hugging Face is the central collaboration platform for the global machine learning community. We provide the infrastructure and ecosystem necessary for researchers, developers, and organizations to create, discover, and share machine learning models, datasets, and applications. Often described as the "Home of Machine Learning," we are dedicated to building the future of AI through openness, accessibility, and community-led innovation.

## Our Platform
We offer an expansive ecosystem designed to power the entire AI lifecycle:
*   **Models & Datasets:** Access a vast library of over 2 million models and 500,000 datasets, enabling developers to build upon the latest advancements in AI.
*   **Hugging Face Spaces:** A powerful environment to build, host, and showcase interactive AI applications.
*   **HuggingChat:** An accessible interface to explore and interact with cutting-edge conversational AI.
*   **Enterprise Solutions:** From inference endpoints and dedicated storage buckets to professional support, we provide scalable, AI-native infrastructure for businesses looking to integrate machine learning into their workflows.

## Our Community & Culture
At Hugging Face, our culture is defined by collaboration. We are a "community-first" organization that believes the best AI is built when brilliant minds work together. Whether it is through our vibrant Discord server, community forums, or the collaborative activity on our Hub, we prioritize knowledge sharing and open-source contributions. 

We are deeply committed to the state of the AI ecosystem, regularly publishing research, insights on global compute landscapes, and educational tracks to ensure that the next generation of builders is equipped to handle the challenges of tomorrow.

## Customers & Partners
Our platform serves a diverse spectrum of users, ranging from individual developers and students to large-scale enterprises. Whether you are a researcher publishing a breakthrough paper, a startup prototyping an agentic workflow, or a corporation requiring high-performance inference, Hugging Face provides the tools to take your project from a raw idea to a deployed, production-ready application.

## Careers: Join Us
Hugging Face is growing, and we are looking for passionate individuals to help us build the future. By joining our team, you become a key part of the world’s most active open-source AI platform. We seek thinkers, engineers, and creators who are motivated by the potential of machine learning to change the world. 

If you are driven by the values of open-source, innovation, and community, we invite you to explore the latest opportunities to help us shape the next era of technology.

### Discover More
*   **Website:** [huggingface.co](https://huggingface.co)
*   **Collaborate:** Join the conversation on our Discord and Forum.
*   **Learn:** Explore our documentation and learning tracks to start your journey today.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>